# Solution: Simple PID Controller


In [1]:
# Import standard libraries
import math
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import numpy as np

In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/bala-c-plus-simplified/bala-c-plus-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"

In [3]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [4]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [5]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [17]:
# Filter and PID coefficients (tune these)
KP = 20.0
KD = 2.0

# When the robot has "tipped over" into an unrecoverable state
TIP_THRESHOLD = math.radians(30)

# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
pitch = 0.0
tipped = False
prev_time = 0.0
with mujoco.viewer.launch_passive(model, data) as viewer:
    # Define free-look camera (control with mouse), looking at robot's back-right
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]
    viewer.cam.distance  = 0.8 
    viewer.cam.azimuth   = 45
    viewer.cam.elevation = -25

    # Simulation loop
    while viewer.is_running():
        step_start = time.time()

        w, x, y, z = data.sensor(IMU_ORIENTATION).data
        pitch = math.atan2(1.0 - 2.0*(x*x+y*y), -2.0*(y*z+w*x))
        pitch_rate = data.sensor(IMU_GYRO).data[0]
    
        ctrl = -(KP*pitch + KD*pitch_rate)
        ctrl = max(-1.0, min(1.0, ctrl))
        data.ctrl[left_motor_id]  = ctrl
        data.ctrl[right_motor_id] = ctrl
        mujoco.mj_step(model, data)
    
        if steps % 20 == 0:
            wv = data.sensor("left_wheel_vel").data[0]
            print(f"t={steps:4d}  pitch={math.degrees(pitch):+6.2f}  rate={pitch_rate:+6.2f}  ctrl={ctrl:+.3f}  wheel={wv:+.2f}")
        
        if abs(pitch) > math.radians(45):
            continue
            print(f"tipped at step {steps}");

        # Render the current simulation state
        viewer.sync()

        # Just print what the sensors say when the robot is upright
        viewer.sync()
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)
        steps += 1

t=   0  pitch= +0.00  rate= +0.00  ctrl=-0.000  wheel=+0.00
t=  20  pitch= +0.51  rate= -0.01  ctrl=-0.163  wheel=-2.27
t=  40  pitch= +0.99  rate= -0.12  ctrl=-0.098  wheel=-3.53
t=  60  pitch= +1.52  rate= +0.17  ctrl=-0.862  wheel=-4.96
t=  80  pitch= +2.11  rate= +0.04  ctrl=-0.817  wheel=-7.50
t= 100  pitch= +3.05  rate= +0.12  ctrl=-1.000  wheel=-9.30
t= 120  pitch= +3.86  rate= +0.49  ctrl=-1.000  wheel=-10.04
t= 140  pitch= +9.42  rate= +1.95  ctrl=-1.000  wheel=-10.16
t= 160  pitch=+24.36  rate= +3.24  ctrl=-1.000  wheel=-10.06


KeyboardInterrupt: 